## Load trained sparse EBMs and data

In [ ]:
import joblib
import copy

folds_ebm = joblib.load("models/folds_ebm.joblib")

fold_0 = folds_ebm[0]
fold_0 = copy.deepcopy(fold_0)

## Add age group column to holdout set, get group fractions and incidences

In [ ]:
from scripts.ebm_fairness import add_age_group_fold, get_incidence_rate_fold

fold_0, group_fractions = add_age_group_fold(fold_0)

print(group_fractions)

incidence_rate = get_incidence_rate_fold(fold_0)

print(incidence_rate)

## Plot cumulative gain curve

In [ ]:
from scripts.ebm_fairness import compute_pred_probs, plot_cumulative_gain_curve, population_share_for_recall

fold_0 = compute_pred_probs(fold_0)

plot_cumulative_gain_curve(fold_0, pos_label=1, n_bins=100)

fraction_needed = population_share_for_recall(fold_0, target_recall=0.8)

print(f"You need to target {fraction_needed:.2%} of the population to capture 80% of positives.")

## Add binary risk bin to fold based on above target

In [ ]:
from scripts.ebm_fairness import add_binary_risk_column

fold_0, threshold = add_binary_risk_column(fold_0, quant=0.51)

print(threshold)

## Get normalized confusion matrices for each age group

In [ ]:
from scripts.ebm_fairness import get_normalized_confusion_matrices_by_group, plot_confusion_matrices_by_group

conf_matrices_prefair = get_normalized_confusion_matrices_by_group(fold_0, 'y_preds', 'alter_gruppen')
plot_confusion_matrices_by_group(conf_matrices_prefair, 'Pre-Fair')

## Apply fairness postprocessing

In [ ]:
from scripts.ebm_fairness import get_postprocess_fit_predict

fold_0 = get_postprocess_fit_predict(fold_0)

conf_matrices_fair = get_normalized_confusion_matrices_by_group(fold_0, 'y_preds_fair', 'alter_gruppen')
plot_confusion_matrices_by_group(conf_matrices_fair, 'Fair')

## Get Accuracy and Coverage of both predictors

In [ ]:
from scripts.ebm_fairness import plot_tpr_positive_accuracy

y_true = fold_0['holdout_Y'].squeeze().to_numpy()

plot_tpr_positive_accuracy(y_true, fold_0['y_preds'], 'pre-Fair')

plot_tpr_positive_accuracy(y_true, fold_0['y_preds_fair'], 'Fair')


